# 03 — OOF + Submission — Modelo de Texto (TF-IDF + SVD + LightGBM)

Extrae features de las descripciones con TF-IDF, reduce con SVD, y entrena LightGBM con KFold para obtener métricas OOF honestas **y** el CSV de submission sobre test.

**Outputs:**
- `text_baseline_oof.csv` → predicciones sobre train (subir a pestaña de práctica)
- `my_team_03.csv` → predicciones sobre test (subir a competencia)

```
participant/
├── submissions/
├── scripts/
└── participant.zip
```

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Rutas
ZIP_PATH              = "/content/drive/MyDrive/participant/participant.zip"
SUBMISSIONS_DRIVE_DIR = "/content/drive/MyDrive/participant/submissions"

os.makedirs(SUBMISSIONS_DRIVE_DIR, exist_ok=True)

In [ ]:
# Extraer CSVs del ZIP
csv_train_path = None
csv_test_path  = None

print("Extrayendo archivos CSV desde el ZIP...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    for f in all_files:
        if f.endswith("train_processed.csv"):
            zip_ref.extract(f, path="/content/")
            csv_train_path = os.path.join("/content", f)
        if f.endswith("test_processed.csv"):
            zip_ref.extract(f, path="/content/")
            csv_test_path = os.path.join("/content", f)

if csv_train_path and csv_test_path:
    print(f"Archivos listos:\n - {csv_train_path}\n - {csv_test_path}")
else:
    raise FileNotFoundError("No se encontraron los archivos CSV dentro del zip.")

Extrayendo archivos CSV desde el ZIP...
Archivos listos:
 - /content/participant/data/tabular/train_processed.csv
 - /content/participant/data/tabular/test_processed.csv


In [ ]:
# Cargar datos
train = pd.read_csv(csv_train_path)
test  = pd.read_csv(csv_test_path)

TARGET = "log_price"

train["description"] = train["description"].fillna("")
test["description"]  = test["description"].fillna("")

print(f"Train: {len(train)} filas | Test: {len(test)} filas")

Train: 11840 filas | Test: 5038 filas


In [ ]:
# TF-IDF + SVD (fit sobre todo el train, transform sobre test)
# Nota: el TF-IDF se fitea sobre todo el train antes del KFold para que
# el vocabulario sea consistente en todos los folds.
tfidf = TfidfVectorizer(
    max_features = 500,
    ngram_range  = (1, 2),
    stop_words   = "english",
    min_df       = 5,
)
train_tfidf = tfidf.fit_transform(train["description"])
test_tfidf  = tfidf.transform(test["description"])

N_COMPONENTS = 50
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
train_svd = svd.fit_transform(train_tfidf)
test_svd  = svd.transform(test_tfidf)

train_svd_df = pd.DataFrame(train_svd)
test_svd_df  = pd.DataFrame(test_svd)

print(f"TF-IDF: {train_tfidf.shape[1]} features → SVD {N_COMPONENTS} dims")

TF-IDF: 500 features → SVD 50 dims


In [ ]:
# Hiperparámetros del modelo
# Modificá estos valores para experimentar
PARAMS = dict(
    n_estimators  = 500,
    learning_rate = 0.05,
    num_leaves    = 63,
    random_state  = 42,
    verbosity     = -1,
)
N_FOLDS = 5

In [ ]:
# OOF Training + predicción sobre test
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

train_oof_txt  = np.zeros(len(train))
test_preds_txt = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_svd_df)):
    X_train_f = train_svd_df.iloc[train_idx]
    X_val_f   = train_svd_df.iloc[val_idx]
    y_train_f = train[TARGET].values[train_idx]
    y_val_f   = train[TARGET].values[val_idx]

    model_txt = lgb.LGBMRegressor(**PARAMS)
    model_txt.fit(X_train_f, y_train_f, eval_set=[(X_val_f, y_val_f)])

    # OOF
    train_oof_txt[val_idx] = model_txt.predict(X_val_f)

    # Test: acumular promedio
    test_preds_txt += model_txt.predict(test_svd_df) / N_FOLDS

    print(f"Fold {fold+1} completado.")

Fold 1 completado.
Fold 2 completado.
Fold 3 completado.
Fold 4 completado.
Fold 5 completado.


In [ ]:
# Métricas OOF
train_price     = train["lastSoldPrice_hpi_adjusted"]
pred_price_oof  = np.expm1(train_oof_txt)

oof_r2   = r2_score(train[TARGET], train_oof_txt)
oof_mae  = mean_absolute_error(train_price, pred_price_oof)
oof_mape = np.mean(np.abs((train_price - pred_price_oof) / train_price)) * 100


print("MÉTRICAS OOF (validación honesta)")
print(f"R²  (log):  {oof_r2:.4f}")
print(f"MAE ($):    ${oof_mae:,.0f}")
print(f"MAPE (%):   {oof_mape:.1f}%")

MÉTRICAS OOF (validación honesta)
R²  (log):  0.3761
MAE ($):    $201,749
MAPE (%):   46.0%


In [ ]:
# Guardar OOF (práctica) y submission (competencia)

# 1. OOF
oof_df = pd.DataFrame({
    "zpid":            train["zpid"],
    "predicted_price": pred_price_oof,
})
oof_file = os.path.join(SUBMISSIONS_DRIVE_DIR, "Lu_text_oof_01.csv")
oof_df.to_csv(oof_file, index=False)

# 2. Test
test_df = pd.DataFrame({
    "zpid":            test["zpid"],
    "predicted_price": np.expm1(test_preds_txt),
})
test_file = os.path.join(SUBMISSIONS_DRIVE_DIR, "Lu_text_test_01.csv")
test_df.to_csv(test_file, index=False)

print(f"OOF guardado en:  {oof_file}  ({len(oof_df)} filas)")
print(f"Test guardado en: {test_file} ({len(test_df)} filas)")

OOF guardado en:  /content/drive/MyDrive/participant/submissions/Lu_text_oof_01.csv  (11840 filas)
Test guardado en: /content/drive/MyDrive/participant/submissions/Lu_text_test_01.csv (5038 filas)
